In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os,sys
# 强制指定你虚拟环境里的 Python 路径（关键！）
os.environ['PYSPARK_PYTHON'] = os.path.join( os.path.dirname(os.getcwd()), ".venv", "Scripts", "python.exe" )
spark=SparkSession.builder\
     .appName('NullSkewFullFlow')\
     .master('local[2]')\
     .config("spark.driver.host", "127.0.0.1")\
     .config("spark.driver.bindAddress", "127.0.0.1") \
     .getOrCreate()
# 原始数据
data_A = [
    (None, "小明"),
    (None, "小红"),
    (None, "小刚"),
    ("1", "小李"),
    ("2", "小王")
]
data_B = [(None, "未知"),
          ("1", "北京"),
          ("2", "上海")]

A = spark.createDataFrame(data_A, ["key", "value"])
B = spark.createDataFrame(data_B, ["key", "city"])
print('----------原始大表A--------------')
A.show()
print('----------原始小表B--------------')
B.show()

----------原始大表A--------------
+----+-----+
| key|value|
+----+-----+
|NULL| 小明|
|NULL| 小红|
|NULL| 小刚|
|   1| 小李|
|   2| 小王|
+----+-----+

----------原始小表B--------------
+----+----+
| key|city|
+----+----+
|NULL|未知|
|   1|北京|
|   2|上海|
+----+----+



In [ ]:
# ========== 1. 打散 null，生成 new_key ==========
A_new = A.withColumn(
    "new_key",
    when(col("key").isNull(), concat(lit("null_"), (rand() * 3).cast("int")))
    .otherwise(col("key"))
)
print("=== 打散后 A_new ===")
A_new.show()
# A_new.printSchema()
# print(A.dtypes)

=== 打散后 A_new ===
+----+-----+-------+
| key|value|new_key|
+----+-----+-------+
|NULL| 小明| null_1|
|NULL| 小红| null_0|
|NULL| 小刚| null_1|
|   1| 小李|      1|
|   2| 小王|      2|
+----+-----+-------+



In [58]:
df1=spark.range(5)
df1.select(rand()).show()



+-------------------------+
|rand(4224936756255767796)|
+-------------------------+
|       0.6605640434901227|
|       0.7989513960684065|
|       0.1622701082461051|
|       0.6336537610329874|
|       0.7600801497588294|
+-------------------------+



In [ ]:
# ========== 2. 小表膨胀 ==========
B_expand = B.crossJoin(spark.range(3).toDF("salt")) \
            .withColumn(
                "new_key",
                when(col("key").isNull(), concat(lit("null_"), col("salt")))
                .otherwise(concat(col("key"), lit("_"), col("salt")))
            )
print("=== 膨胀后小表 B_expand ===")
B_expand.show(truncate=False)   #truncate=False = 完整显示，不省略

=== 膨胀后小表 B_expand ===
+----+----+----+-------+
|key |city|salt|new_key|
+----+----+----+-------+
|NULL|未知|0   |null_0 |
|NULL|未知|1   |null_1 |
|NULL|未知|2   |null_2 |
|1   |北京|0   |1_0    |
|1   |北京|1   |1_1    |
|1   |北京|2   |1_2    |
|2   |上海|0   |2_0    |
|2   |上海|1   |2_1    |
|2   |上海|2   |2_2    |
+----+----+----+-------+



In [10]:
# ========== 3. 打散后 join ==========
result = A_new.join(B_expand, on="new_key").select(A_new.key, "value", "city")
print("=== 最终 join 结果 ===")
result.show()

=== 最终 join 结果 ===
+----+-----+----+
| key|value|city|
+----+-----+----+
|NULL| 小红|未知|
|NULL| 小明|未知|
|NULL| 小刚|未知|
+----+-----+----+

